# 04 – Isolation Forest and Extreme Learning Machine (corrected)

Replaces the Isolation Forest part of `04_feature_analysis_comparison.ipynb` and `elm_experiment.ipynb`.

**What changed**
1. Both models use **the same split** as the autoencoder.
2. **Scalers are fitted on training data only.** The original ELM notebook fitted the scaler on all rows before splitting.
3. **The Isolation Forest is evaluated on the test set with the same threshold rule as the autoencoder** (percentiles of held-out normal traffic), as well as with `contamination=0.05`.
4. The ELM is still **supervised**: it is trained on labelled attacks, so its scores are not comparable like-for-like with the unsupervised models.

In [1]:
import os, random, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)
from hpelm import ELM

In [2]:
df = pd.read_csv("features_conn.csv")
split = df.pop("split")
capture = df.pop("capture")
y = df.pop("label").astype(int)
feature_names = df.columns.tolist()
X = df.values.astype("float32")

tr, va, te = (split == "train").values, (split == "val_normal").values, (split == "test").values
print("train:", tr.sum(), " val (normal only):", va.sum(), " test:", te.sum())
print("test class balance:", y[te].value_counts().to_dict())

train: 101467  val (normal only): 3105  test: 26143
test class balance: {1: 22262, 0: 3881}


## Isolation Forest (unsupervised, trained on all training traffic without labels)

In [3]:
X_train_all = X[tr]
scaler_if = StandardScaler().fit(X_train_all)
iforest = IsolationForest(n_estimators=200, contamination=0.05, random_state=SEED, n_jobs=-1)
iforest.fit(scaler_if.transform(X_train_all))

Xte, y_test = scaler_if.transform(X[te]), y[te].values
if_score = -iforest.score_samples(Xte)
val_score = -iforest.score_samples(scaler_if.transform(X[va]))

if_auc, if_ap = roc_auc_score(y_test, if_score), average_precision_score(y_test, if_score)
print(f"Isolation Forest ROC-AUC: {if_auc:.4f}   PR-AUC: {if_ap:.4f}")

def metrics(name, pred):
    cm = confusion_matrix(y_test, pred)
    return {"model": name, "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred),
            "false_positive_rate": cm[0, 1] / cm[0].sum()}

rows = [metrics("Isolation Forest (contamination=0.05)", (iforest.predict(Xte) == -1).astype(int))]
for p in [95, 90, 85, 80]:
    rows.append(metrics(f"Isolation Forest ({p}th pct)", (if_score > np.percentile(val_score, p)).astype(int)))
if_table = pd.DataFrame(rows)
if_table.round(3)

Isolation Forest ROC-AUC: 0.1431   PR-AUC: 0.7269


,model,precision,recall,f1,false_positive_rate
0,Isolation Forest (contamination=0.05),0.513,0.033,0.062,0.179
1,Isolation Forest (95th pct),0.355,0.005,0.010,0.052
2,Isolation Forest (90th pct),0.411,0.013,0.025,0.107
3,Isolation Forest (85th pct),0.568,0.032,0.061,0.141
4,Isolation Forest (80th pct),0.523,0.038,0.071,0.199


### Why the Isolation Forest scores so badly here

Isolation Forest assumes anomalies are **rare**. In this data, 88% of training connections are attacks, so to the forest the attack pattern *is* normal and the minority normal traffic looks anomalous. That is why its ROC-AUC is far below 0.5. The thesis figures for Isolation Forest (precision 0.94, recall 0.91) cannot be reproduced.

The fair comparison with the autoencoder is to train the forest the same way: on normal traffic only.

In [4]:
scaler_ifn = StandardScaler().fit(X[tr & (y.values == 0)])
iforest_n = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1)
iforest_n.fit(scaler_ifn.transform(X[tr & (y.values == 0)]))
ifn_score = -iforest_n.score_samples(scaler_ifn.transform(X[te]))
ifn_val = -iforest_n.score_samples(scaler_ifn.transform(X[va]))
ifn_auc, ifn_ap = roc_auc_score(y_test, ifn_score), average_precision_score(y_test, ifn_score)
print(f"Isolation Forest (normal-only training) ROC-AUC: {ifn_auc:.4f}   PR-AUC: {ifn_ap:.4f}")

rows_n = [metrics(f"Isolation Forest, normal-only ({p}th pct)", (ifn_score > np.percentile(ifn_val, p)).astype(int))
          for p in [95, 90, 85, 80]]
if_table = pd.concat([if_table, pd.DataFrame(rows_n)], ignore_index=True)
if_table.round(3)

Isolation Forest (normal-only training) ROC-AUC: 0.7669   PR-AUC: 0.9264


,model,precision,recall,f1,false_positive_rate
0,Isolation Forest (contamination=0.05),0.513,0.033,0.062,0.179
1,Isolation Forest (95th pct),0.355,0.005,0.010,0.052
2,Isolation Forest (90th pct),0.411,0.013,0.025,0.107
3,Isolation Forest (85th pct),0.568,0.032,0.061,0.141
4,Isolation Forest (80th pct),0.523,0.038,0.071,0.199
5,"Isolation Forest, normal-only (95th pct)",0.938,0.151,0.260,0.057
6,"Isolation Forest, normal-only (90th pct)",0.903,0.174,0.292,0.107
7,"Isolation Forest, normal-only (85th pct)",0.903,0.241,0.381,0.149
8,"Isolation Forest, normal-only (80th pct)",0.911,0.358,0.514,0.200


## Extreme Learning Machine (supervised)

In [5]:
X_tr, y_tr = X[tr | va], y[tr | va].values   # all training rows, labelled
scaler_elm = StandardScaler().fit(X_tr)
Xtr_s, Xte_s = scaler_elm.transform(X_tr), scaler_elm.transform(X[te])

np.random.seed(SEED)
elm = ELM(Xtr_s.shape[1], 2)
elm.add_neurons(256, "sigm")
T = np.zeros((len(y_tr), 2)); T[np.arange(len(y_tr)), y_tr] = 1
elm.train(Xtr_s, T, "c")

elm_prob = elm.predict(Xte_s)
elm_pred = np.argmax(elm_prob, axis=1)
elm_score = elm_prob[:, 1] - elm_prob[:, 0]
print(confusion_matrix(y_test, elm_pred))
elm_row = metrics("ELM (supervised)", elm_pred)
elm_row["roc_auc"] = roc_auc_score(y_test, elm_score)
pd.DataFrame([elm_row]).round(4)

[[ 3720   161]
 [  161 22101]]


,model,precision,recall,f1,false_positive_rate,roc_auc
0,ELM (supervised),0.9928,0.9928,0.9928,0.0415,0.9963


In [6]:
pd.DataFrame({"row_id": np.where(te)[0], "label": y_test, "if_score": if_score, "ifn_score": ifn_score, "elm_score": elm_score}).to_csv("results/if_elm_test_scores.csv", index=False)
if_table.to_csv("results/if_thresholds.csv", index=False)
json.dump({"if_roc_auc": if_auc, "if_pr_auc": if_ap, "ifn_roc_auc": ifn_auc, "ifn_pr_auc": ifn_ap,
           "ifn_val_percentiles": {p: float(np.percentile(ifn_val, p)) for p in [80, 85, 90, 95]},
           "if_val_percentiles": {p: float(np.percentile(val_score, p)) for p in [80, 85, 90, 95]},
           "elm": elm_row}, open("results/if_elm_summary.json", "w"), indent=2, default=float)
print("Saved results/if_* and results/if_elm_*")

Saved results/if_* and results/if_elm_*
